# BART meeting summarization fine-tuning

This notebook documents Hugging Face BART fine-tuning workflow for meeting summarization. It loads a public conversational dataset, tokenizes transcript/summary pairs, evaluates with ROUGE, and saves the resulting model when training completes.


# Environment and dependencies

In [ ]:

# ============================================================================
# CELL 1: GPU Check and Dependencies Installation
# ============================================================================

import sys
print("="*70)
print("MEETING SUMMARIZATION - BART FINE-TUNING")
print("="*70)
print("\n[STEP 1/12] Checking environment...\n")

# Check GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"✅ GPU Memory: {gpu_memory:.1f} GB")
    print("✅ Training will be fast!\n")
else:
    print("⚠️  WARNING: No GPU detected!")
    print("⚠️  Training will be VERY slow (8-12 hours)")
    print("⚠️  Please enable GPU: Runtime → Change runtime type → GPU\n")
    response = input("Continue without GPU? (yes/no): ")
    if response.lower() != 'yes':
        print("Please enable GPU and restart.")
        sys.exit()

print("-"*70)
print("Installing required packages...")
print("(This takes 2-3 minutes)\n")

# Install packages quietly
!pip install -q transformers==4.36.0
!pip install -q datasets==2.16.0
!pip install -q evaluate==0.4.1
!pip install -q rouge-score==0.1.2
!pip install -q accelerate==0.21.0
!pip install -q peft==0.6.0
!pip install -q sentencepiece==0.1.99

print("✅ All packages installed successfully!\n")

MEETING SUMMARIZATION - BART FINE-TUNING

[STEP 1/12] Checking environment...

✅ GPU Available: Tesla T4
✅ GPU Memory: 15.8 GB
✅ Training will be fast!

----------------------------------------------------------------------
Installing required packages...
(This takes 2-3 minutes)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 7.1 MB/s eta 0:00:00
✅ All packages installed successfully!



In [ ]:
# ============================================================================
# CELL 2: Mount Google Drive
# ============================================================================

print("="*70)
print("[STEP 2/12] Connecting to Google Drive")
print("="*70)
print("\nYour trained model will be saved to Google Drive.")
print("This allows you to access it anytime.\n")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create save directory
import os
save_dir = '/content/drive/MyDrive/meeting-bart-finetuned'
os.makedirs(save_dir, exist_ok=True)

print(f"\n✅ Google Drive connected!")
print(f"✅ Save location: {save_dir}\n")


[STEP 2/12] Connecting to Google Drive

Your trained model will be saved to Google Drive.
This allows you to access it anytime.

Mounted at /content/drive

✅ Google Drive connected!
✅ Save location: /content/drive/MyDrive/meeting-bart-finetuned



# Dataset loading

The loader first attempts AMI and then falls back to DialogSum or SAMSum. Verify that the selected dataset contains distinct transcript and summary fields before training.

In [ ]:

# ============================================================================
# CELL 3: Load AMI Meeting Dataset
# ============================================================================

print("="*70)
print("[STEP 3/12] Loading AMI Meeting Corpus")
print("="*70)

from datasets import load_dataset

print("Downloading AMI dataset...")
print("(First time: 5-10 minutes, then cached)\n")

try:
    # Load AMI Meeting Corpus
    dataset = load_dataset("edinburghcstr/ami", "ihm")

    print("✅ AMI Meeting Corpus loaded successfully!\n")

    # Dataset info
    print("-"*70)
    print("DATASET INFORMATION")
    print("-"*70)
    print(f"Training meetings:   {len(dataset['train'])}")
    print(f"Validation meetings: {len(dataset['validation'])}")
    print(f"Test meetings:       {len(dataset['test'])}")
    print(f"\nTotal: ~100 hours of meeting recordings\n")

    # Show sample
    print("-"*70)
    print("SAMPLE MEETING PREVIEW")
    print("-"*70)
    sample = dataset['train'][0]

    # AMI has different column names, check what's available
    print("\nAvailable columns:", dataset['train'].column_names)

    # Get transcript (column name may vary)
    if 'transcript' in dataset['train'].column_names:
        transcript_col = 'transcript'
    elif 'text' in dataset['train'].column_names:
        transcript_col = 'text'
    elif 'dialogue' in dataset['train'].column_names:
        transcript_col = 'dialogue'
    else:
        transcript_col = dataset['train'].column_names[0]

    # Get summary column
    if 'summary' in dataset['train'].column_names:
        summary_col = 'summary'
    elif 'extractive_summary' in dataset['train'].column_names:
        summary_col = 'extractive_summary'
    else:
        # Create summary from transcript (first 200 chars as fallback)
        summary_col = transcript_col

    print(f"\nUsing columns: {transcript_col} → {summary_col}")
    print(f"\nMeeting excerpt:\n{str(sample[transcript_col])[:400]}...\n")

    if transcript_col != summary_col:
        print(f"Summary:\n{str(sample[summary_col])[:300]}...\n")

    dataset_loaded = True

except Exception as e:
    print(f"❌ Failed to load AMI: {e}\n")
    print("Trying alternative: DialogSum (meeting conversations)...\n")

    try:
        dataset = load_dataset("knkarthick/dialogsum")
        transcript_col = 'dialogue'
        summary_col = 'summary'

        print("✅ DialogSum loaded successfully!")
        
        print("-"*70)
        print("DATASET INFORMATION")
        print("-"*70)
        print(f"Training samples:   {len(dataset['train'])}")
        print(f"Validation samples: {len(dataset['validation'])}")
        print(f"Test samples:       {len(dataset['test'])}")

        dataset_loaded = True

    except Exception as e2:
        print(f"❌ Failed to load DialogSum: {e2}\n")
        print("Using final fallback: SAMSum...\n")

        dataset = load_dataset("samsum")
        transcript_col = 'dialogue'
        summary_col = 'summary'

        print("✅ SAMSum loaded (conversations)\n")
        dataset_loaded = True

# Optional: Use smaller subset for faster testing
# Uncomment these lines to train on subset (faster, for testing)
USE_SUBSET = False  # Change to True for quick test

if USE_SUBSET:
    print("\n⚠️  USING SUBSET FOR QUICK TESTING")
    print("(Change USE_SUBSET = False for full training)\n")
    dataset['train'] = dataset['train'].select(range(500))
    dataset['validation'] = dataset['validation'].select(range(100))
    dataset['test'] = dataset['test'].select(range(50))
    print(f"Subset size: {len(dataset['train'])} train, {len(dataset['validation'])} val\n")

print("="*70 + "\n")

[STEP 3/12] Loading AMI Meeting Corpus

AMI is a dataset of 100 hours of real business meetings.
Perfect for your meeting intelligence project!

(First time: 5-10 minutes, then cached)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1429: FutureWarning: The repository for edinburghcstr/ami contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/edinburghcstr/ami
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warni

✅ AMI Meeting Corpus loaded successfully!

----------------------------------------------------------------------
DATASET INFORMATION
----------------------------------------------------------------------
Training meetings:   108502
Validation meetings: 13098
Test meetings:       12643

Total: ~100 hours of meeting recordings

----------------------------------------------------------------------
SAMPLE MEETING PREVIEW
----------------------------------------------------------------------

Available columns: ['meeting_id', 'audio_id', 'text', 'audio', 'begin_time', 'end_time', 'microphone_id', 'speaker_id']

Using columns: text → text

Meeting excerpt:
IF YOU IF YOU S. S. H. AND THEY HAVE THIS BIG WARNING ABOUT DOING NOTHING AT ALL IN THE GATEWAY MACHINE...




# Base model

initialize `facebook/bart-base`; this is the pretrained starting point for the Trainer run.

In [ ]:

# ============================================================================
# CELL 4: Load BART Model
# ============================================================================

print("="*70)
print("[STEP 4/12] Loading BART Model")
print("="*70)

from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-base"

print(f"Loading {model_name}...")
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# Model info
param_count = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Model loaded: {param_count:.0f}M parameters")

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Model on device: {device}\n")


[STEP 4/12] Loading BART Model

BART-base: 140M parameters, good balance of speed and quality



/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

Loading facebook/bart-base...
✅ Model loaded: 139M parameters
✅ Model on device: cuda



# Preprocessing

Inputs are truncated/padded to 1,024 tokens and targets to 256 tokens. The current dataset fallback may select the transcript as both input and target when no summary field exists.

In [ ]:

# ============================================================================
# CELL 5: Preprocess Dataset
# ============================================================================

print("="*70)
print("[STEP 5/12] Preprocessing Dataset")
print("="*70)
print("\nTokenizing meetings and summaries...")

def preprocess_function(examples):
    """
    Convert meeting transcripts to model inputs
    """
    # Get inputs (meeting transcripts)
    inputs = examples[transcript_col]

    # Get targets (summaries)
    targets = examples[summary_col]

    # Tokenize inputs (meetings can be long)
    model_inputs = tokenizer(
        inputs,
        max_length=1024,      # Max meeting length
        truncation=True,       # Cut if longer
        padding="max_length"   # Pad to same size
    )

    # Tokenize targets (summaries are shorter)
    labels = tokenizer(
        targets,
        max_length=256,        # Max summary length
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply to dataset
print("Tokenizing...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Processing meetings",
    num_proc=2  # Parallel processing
)

print(f"✅ Tokenization complete!")
print(f"✅ Ready: {len(tokenized_dataset['train'])} training samples\n")


[STEP 5/12] Preprocessing Dataset

Tokenizing meetings and summaries...
(This takes 5-10 minutes)

Tokenizing...
✅ Tokenization complete!
✅ Ready: 108502 training samples



# Evaluation metrics

ROUGE-1, ROUGE-2, and ROUGE-L are computed from generated predictions and reference labels.

In [ ]:

# ============================================================================
# CELL 6: Setup Evaluation Metrics
# ============================================================================

print("="*70)
print("[STEP 6/12] Configuring Evaluation Metrics")
print("="*70)
print("\nUsing ROUGE scores (standard for summarization)\n")

import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """
    Calculate ROUGE scores
    """
    predictions, labels = eval_pred

    # Remove None and non-int tokens
    clean_preds = []
    for pred in predictions:
        clean_pred = [p for p in pred if p is not None and isinstance(p, (int, np.integer))]
        clean_preds.append(clean_pred)

    decoded_preds = tokenizer.batch_decode(
        clean_preds,
        skip_special_tokens=True
    )


    # Decode labels
    clean_labels = []
    for label in labels:
        clean_label = [l for l in label if l is not None and l != -100 and isinstance(l, (int, np.integer))]
        clean_labels.append(clean_label)

    decoded_labels = tokenizer.batch_decode(
        clean_labels,
        skip_special_tokens=True
    )


    # Compute ROUGE
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

print("✅ Metrics configured!\n")


[STEP 6/12] Configuring Evaluation Metrics

Using ROUGE scores (standard for summarization)

✅ Metrics configured!



# Training configuration

The Trainer uses epoch-level evaluation and checkpoint saving with the hyperparameters shown below.

In [ ]:

# ============================================================================
# CELL 7: Training Configuration
# ============================================================================

print("="*70)
print("[STEP 7/12] Training Configuration")
print("="*70 + "\n")

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    # Output directory
    output_dir="./results",

    # Evaluation and saving
    evaluation_strategy="epoch",
    save_strategy="epoch",

    # Training hyperparameters
    learning_rate=3e-5,
    per_device_train_batch_size=4,    # Reduce to 2 if out of memory
    per_device_eval_batch_size=4,
    num_train_epochs=3,                # 3 epochs is good
    weight_decay=0.01,

    # Warmup
    warmup_steps=500,

    # Generation (for evaluation)
    predict_with_generate=True,
    generation_max_length=256,

    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,

    # Efficiency
    fp16=torch.cuda.is_available(),    # Mixed precision if GPU
    save_total_limit=2,                # Keep only 2 checkpoints

    # Logging
    logging_dir="./logs",
    logging_steps=100,
    logging_first_step=True,

    # Disable external services
    report_to="none",
    push_to_hub=False,
)

print("Training Configuration:")
print(f"  • Epochs: {training_args.num_train_epochs}")
print(f"  • Batch size: {training_args.per_device_train_batch_size}")
print(f"  • Learning rate: {training_args.learning_rate}")
print(f"  • Mixed precision: {training_args.fp16}")
print(f"  • Warmup steps: {training_args.warmup_steps}")
print(f"  • Save strategy: After each epoch\n")


[STEP 7/12] Training Configuration

Training Configuration:
  • Epochs: 3
  • Batch size: 4
  • Learning rate: 3e-05
  • Mixed precision: True
  • Warmup steps: 500
  • Save strategy: After each epoch



# Trainer setup and execution

The saved run reached `trainer.train()` but failed before producing a completed training result.

In [ ]:

# ============================================================================
# CELL 8: Create Trainer
# ============================================================================

print("="*70)
print("[STEP 8/12] Creating Trainer")
print("="*70 + "\n")

from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer created and ready!\n")


[STEP 8/12] Creating Trainer

✅ Trainer created and ready!



/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:427: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:

# ============================================================================
# CELL 9: START TRAINING
# ============================================================================

print("="*70)
print("[STEP 9/12] STARTING TRAINING")
print("="*70)
print("\n🚀 Training begins now!\n")

# Estimate time
samples = len(tokenized_dataset["train"])
batch_size = training_args.per_device_train_batch_size
steps_per_epoch = samples // batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

if torch.cuda.is_available():
    time_estimate = "1-2 hours"
else:
    time_estimate = "8-12 hours"

print(f"Dataset size: {samples} meetings")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total epochs: {training_args.num_train_epochs}")
print(f"Estimated time: {time_estimate}")
print("\nProgress will be displayed below...")
print("="*70 + "\n")

# START TRAINING
try:
    train_result = trainer.train()

    print("\n" + "="*70)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("="*70)
    print(f"\nFinal training loss: {train_result.training_loss:.4f}\n")

except Exception as e:
    print(f"\n❌ Training failed: {e}")
    print("\nTroubleshooting:")
    print("1. Out of memory? Reduce batch_size to 2 or 1")
    print("2. Session disconnected? Training auto-saves checkpoints")
    print("3. Other error? Check error message above\n")
    raise


[STEP 9/12] STARTING TRAINING

🚀 Training begins now!

Dataset size: 108502 meetings
Steps per epoch: 27125
Total epochs: 3
Estimated time: 1-2 hours

Progress will be displayed below...



/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:1301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  new_forward = torch.cuda.amp.autocast(dtype=torch.float16)(model_forward_func)


Epoch,Training Loss,Validation Loss



❌ Training failed: sequence item 18: expected str instance, NoneType found

Troubleshooting:
1. Out of memory? Reduce batch_size to 2 or 1
2. Session disconnected? Training auto-saves checkpoints
3. Other error? Check error message above



TypeError: sequence item 18: expected str instance, NoneType found

# Post-training evaluation and export


In [ ]:

# ============================================================================
# CELL 10: Evaluate on Test Set
# ============================================================================

print("="*70)
print("[STEP 10/12] Evaluating on Test Set")
print("="*70 + "\n")

print("Running evaluation on test meetings...\n")

eval_results = trainer.evaluate(tokenized_dataset["test"])

print("-"*70)
print("TEST SET RESULTS")
print("-"*70)
for key, value in eval_results.items():
    if key.startswith("eval_"):
        metric_name = key.replace("eval_", "").upper()
        print(f"{metric_name:20s}: {value:.4f}")
print("-"*70 + "\n")



In [ ]:

# ============================================================================
# CELL 11: Test with Real Example
# ============================================================================

print("="*70)
print("[STEP 11/12] Testing with Example Meeting")
print("="*70 + "\n")

# Get test sample
test_sample = dataset["test"][0]

print("="*70)
print("ORIGINAL MEETING")
print("="*70)
meeting_text = str(test_sample[transcript_col])
print(meeting_text[:600] + "...\n" if len(meeting_text) > 600 else meeting_text + "\n")

if transcript_col != summary_col:
    print("="*70)
    print("REFERENCE SUMMARY")
    print("="*70)
    print(test_sample[summary_col], "\n")

print("="*70)
print("GENERATED SUMMARY (Your Fine-tuned Model)")
print("="*70)

# Generate summary
inputs = tokenizer(
    meeting_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=256,
        min_length=50,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(generated, "\n")


In [ ]:

# ============================================================================
# CELL 12: Save Model
# ============================================================================

print("="*70)
print("[STEP 12/12] Saving Your Fine-tuned Model")
print("="*70 + "\n")

print("Saving to Google Drive...")
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model saved to: {save_dir}\n")

# Also save locally
local_dir = "./meeting-bart-finetuned"
trainer.save_model(local_dir)
tokenizer.save_pretrained(local_dir)
print(f"✅ Also saved locally to: {local_dir}\n")

print("="*70)
print("MODEL EXPORT COMPLETED")
print("="*70)
print("\nYour fine-tuned model is ready to use!")
print(f"\nLocation: {save_dir}")
print("\nNext steps:")
print("  1. Download model from Google Drive")
print("  2. Use in your meeting intelligence system")
print("  3. Deploy to Hugging Face Spaces")
print("\n" + "="*70)


# Loading a saved model for inference

This example shows how the exported model would be loaded by the application.

In [ ]:

# ============================================================================
# BONUS: Model Usage Instructions
# ============================================================================

print("\n" + "="*70)
print("HOW TO USE YOUR MODEL")
print("="*70)
print("""
In your project, load the model like this:

from transformers import BartTokenizer, BartForConditionalGeneration

# Load your fine-tuned model
tokenizer = BartTokenizer.from_pretrained("./meeting-bart-finetuned")
model = BartForConditionalGeneration.from_pretrained("./meeting-bart-finetuned")

# Summarize a meeting
def summarize_meeting(transcript):
    inputs = tokenizer(transcript, max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = model.generate(inputs["input_ids"], max_length=256, num_beams=4)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Use it
summary = summarize_meeting("Your meeting transcript here...")
print(summary)
""")

print("="*70)
print("\n" + "="*70)